# Week 5 - Machine Learning Model Design and Implementation

## Objective

Build two ML routes on top of the existing project: (1) predict forward JPM volatility and insert it into the validated chooser BSM model, and (2) predict the model-derived chooser proxy price directly. A 20-trading-day purge separates the chronological 70%/15%/15% blocks before Week 6 tuning.

## 1. Load Week 2 Features and Week 4 Pricing Engine

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=UserWarning)


def locate_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path(r"G:\JPM-Chooser Option Pricing")]
    for candidate in candidates:
        if (candidate / "Week2" / "processed_data" / "market_data_processed.csv").exists() and (candidate / "Week 4" / "bsm_chooser.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Week 2 dataset and Week 4 pricing engine.")


PROJECT_ROOT = locate_project_root()
WEEK_DIR = Path.cwd()
RESULTS_DIR = WEEK_DIR / "model_results"
FIGURES_DIR = WEEK_DIR / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(WEEK_DIR))
sys.path.insert(0, str(PROJECT_ROOT / "Week 4"))

from bsm_chooser import simple_chooser_price
from ml_features import BASE_FEATURES, PRICING_FEATURES, VOLATILITY_FEATURES, build_ml_dataset, chronological_split
from ml_pipeline import build_initial_models

with open(WEEK_DIR / "ml_config.json", encoding="utf-8") as stream:
    CONFIG = json.load(stream)

market = pd.read_csv(PROJECT_ROOT / "Week2" / "processed_data" / "market_data_processed.csv", parse_dates=["Date"])
print(f"Loaded {len(market):,} Week 2 observations.")

Loaded 1,760 Week 2 observations.


## 2. Leakage-Safe Feature and Target Preparation

In [2]:
contract = CONFIG["contract"]
dataset = build_ml_dataset(
    market,
    simple_chooser_price,
    strike=contract["strike"],
    dividend_yield=contract["dividend_yield"],
    choice_time=contract["choice_time_years"],
    maturity=contract["maturity_years"],
    horizon=CONFIG["target_horizon_trading_days"],
)
dataset = chronological_split(
    dataset, CONFIG["split"]["train"], CONFIG["split"]["validation"],
    CONFIG["split"]["purge_horizon_trading_days"],
)

print(f"Usable ML rows: {len(dataset):,}")
print(f"Volatility features: {len(VOLATILITY_FEATURES)}")
print(f"Pricing features: {len(PRICING_FEATURES)}")
dataset[["Date", "Split", "Target_Forward_Volatility_20D", "Target_Chooser_Proxy_Price"]].head()

Usable ML rows: 1,680
Volatility features: 17
Pricing features: 19


        Date  Split  Target_Forward_Volatility_20D  Target_Chooser_Proxy_Price
0 2018-03-29  train                       0.241277                   60.523086
1 2018-04-02  train                       0.231287                   62.128619
2 2018-04-03  train                       0.226171                   60.890507
3 2018-04-04  train                       0.220514                   59.599375
4 2018-04-05  train                       0.214617                   58.405200

## 3. Chronological 70% / 15% / 15% Blocks with a 20-Day Purge

In [3]:
split_summary = dataset.groupby("Split", sort=False).agg(
    Observations=("Date", "size"),
    Start_Date=("Date", "min"),
    End_Date=("Date", "max"),
    Mean_Target_Volatility=("Target_Forward_Volatility_20D", "mean"),
    Mean_Target_Price=("Target_Chooser_Proxy_Price", "mean"),
).reset_index()

assert split_summary["Observations"].sum() == len(dataset)
assert dataset["Date"].is_monotonic_increasing
train_check = dataset[dataset["Split"] == "train"]
validation_check = dataset[dataset["Split"] == "validation"]
test_check = dataset[dataset["Split"] == "test"]
assert train_check["Target_End_Date"].max() < validation_check["Date"].min()
assert validation_check["Target_End_Date"].max() < test_check["Date"].min()
split_summary

                     Split  ...  Mean_Target_Price
0                    train  ...          49.372763
1  purged_train_validation  ...          29.449909
2               validation  ...          23.884524
3   purged_validation_test  ...          11.857900
4                     test  ...          45.465257

[5 rows x 6 columns]

## 4. Feature Dictionary

In [4]:
descriptions = {
    "Daily_Return": "Current one-day JPM simple return",
    "Abs_Return_1D": "Absolute one-day return",
    "Rolling_Volatility_5D": "Past 5-day annualized JPM volatility",
    "Rolling_Volatility_10D": "Past 10-day annualized JPM volatility",
    "Rolling_Volatility_20D": "Past 20-day annualized JPM volatility",
    "Rolling_Volatility_60D": "Past 60-day annualized JPM volatility",
    "VIX_Close": "Cboe VIX closing level",
    "VIX_Return": "One-day VIX return",
    "VIX_JPM_Correlation_20D": "Past 20-day correlation of JPM and VIX returns",
    "Treasury_Rate_Decimal": "Treasury proxy converted from percent to decimal",
    "Interest_Rate_Momentum": "One-day Treasury-rate change",
    "Price_Momentum_20D": "JPM 20-day price momentum",
    "MA20_Gap": "JPM price relative to 20-day moving average",
    "MA50_Gap": "JPM price relative to 50-day moving average",
    "Intraday_Range": "Daily high-low range divided by close",
    "Overnight_Gap": "Open relative to previous close",
    "Volume_ZScore_20D": "Trading volume standardized over trailing 20 days",
    "Close": "Current adjusted JPM closing price",
    "Log_Moneyness": "Log of current JPM close divided by the fixed strike",
}
feature_dictionary = pd.DataFrame([
    {"Feature": feature, "Description": descriptions[feature], "Look_Ahead": False, "Source": "Week 2 or trailing derivation"}
    for feature in PRICING_FEATURES
])
feature_dictionary

                    Feature  ...                         Source
0              Daily_Return  ...  Week 2 or trailing derivation
1             Abs_Return_1D  ...  Week 2 or trailing derivation
2     Rolling_Volatility_5D  ...  Week 2 or trailing derivation
3    Rolling_Volatility_10D  ...  Week 2 or trailing derivation
4    Rolling_Volatility_20D  ...  Week 2 or trailing derivation
5    Rolling_Volatility_60D  ...  Week 2 or trailing derivation
6                 VIX_Close  ...  Week 2 or trailing derivation
7                VIX_Return  ...  Week 2 or trailing derivation
8   VIX_JPM_Correlation_20D  ...  Week 2 or trailing derivation
9     Treasury_Rate_Decimal  ...  Week 2 or trailing derivation
10   Interest_Rate_Momentum  ...  Week 2 or trailing derivation
11       Price_Momentum_20D  ...  Week 2 or trailing derivation
12                 MA20_Gap  ...  Week 2 or trailing derivation
13                 MA50_Gap  ...  Week 2 or trailing derivation
14           Intraday_Range  ...  Week 2

## 5. Initial Approach 1 Models: Volatility Prediction + BSM

In [5]:
def metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    }


train = dataset[dataset["Split"] == "train"]
validation = dataset[dataset["Split"] == "validation"]
X_train_vol, X_validation_vol = train[VOLATILITY_FEATURES], validation[VOLATILITY_FEATURES]

initial_rows = []
volatility_predictions = {}
for name, model in build_initial_models(CONFIG["random_state"]).items():
    model.fit(X_train_vol, train["Target_Forward_Volatility_20D"])
    predicted_volatility = np.clip(model.predict(X_validation_vol), 0.01, 2.0)
    volatility_predictions[name] = predicted_volatility
    predicted_price = simple_chooser_price(
        validation["Close"], contract["strike"], validation["Treasury_Rate_Decimal"],
        contract["dividend_yield"], predicted_volatility,
        contract["choice_time_years"], contract["maturity_years"],
    )
    initial_rows.append({"Approach": "1 - ML volatility + BSM", "Model": name, "Target": "Forward volatility", **metrics(validation["Target_Forward_Volatility_20D"], predicted_volatility)})
    initial_rows.append({"Approach": "1 - ML volatility + BSM", "Model": name, "Target": "Chooser proxy price", **metrics(validation["Target_Chooser_Proxy_Price"], predicted_price)})

pd.DataFrame(initial_rows)

                  Approach              Model  ...      RMSE        R2
0  1 - ML volatility + BSM  Linear Regression  ...  0.078004 -0.143607
1  1 - ML volatility + BSM  Linear Regression  ...  5.815108  0.412210
2  1 - ML volatility + BSM      Random Forest  ...  0.074392 -0.040151
3  1 - ML volatility + BSM      Random Forest  ...  5.372266  0.498326
4  1 - ML volatility + BSM     Histogram GBDT  ...  0.074663 -0.047734
5  1 - ML volatility + BSM     Histogram GBDT  ...  5.499832  0.474218
6  1 - ML volatility + BSM     Neural Network  ...  0.119055 -1.664027
7  1 - ML volatility + BSM     Neural Network  ...  9.139376 -0.451911

[8 rows x 6 columns]

## 6. Initial Approach 2 Models: Direct Supervised Proxy Pricing

In [6]:
direct_predictions = {}
X_train_price, X_validation_price = train[PRICING_FEATURES], validation[PRICING_FEATURES]
for name, model in build_initial_models(CONFIG["random_state"]).items():
    model.fit(X_train_price, train["Target_Chooser_Proxy_Price"])
    predicted_price = np.clip(model.predict(X_validation_price), 0.0, None)
    direct_predictions[name] = predicted_price
    initial_rows.append({"Approach": "2 - Direct proxy pricing", "Model": name, "Target": "Chooser proxy price", **metrics(validation["Target_Chooser_Proxy_Price"], predicted_price)})

initial_metrics = pd.DataFrame(initial_rows).sort_values(["Target", "RMSE"]).reset_index(drop=True)
initial_metrics

                    Approach              Model  ...       RMSE         R2
0    1 - ML volatility + BSM      Random Forest  ...   5.372266   0.498326
1   2 - Direct proxy pricing  Linear Regression  ...   5.483655   0.477306
2    1 - ML volatility + BSM     Histogram GBDT  ...   5.499832   0.474218
3    1 - ML volatility + BSM  Linear Regression  ...   5.815108   0.412210
4    1 - ML volatility + BSM     Neural Network  ...   9.139376  -0.451911
5   2 - Direct proxy pricing     Histogram GBDT  ...   9.869078  -0.693012
6   2 - Direct proxy pricing      Random Forest  ...  10.169062  -0.797499
7   2 - Direct proxy pricing     Neural Network  ...  27.731331 -12.367444
8    1 - ML volatility + BSM      Random Forest  ...   0.074392  -0.040151
9    1 - ML volatility + BSM     Histogram GBDT  ...   0.074663  -0.047734
10   1 - ML volatility + BSM  Linear Regression  ...   0.078004  -0.143607
11   1 - ML volatility + BSM     Neural Network  ...   0.119055  -1.664027

[12 rows x 6 columns]

## 7. Save Week 5 Deliverables

In [7]:
dataset.to_csv(RESULTS_DIR / "ml_dataset.csv", index=False)
split_summary.to_csv(RESULTS_DIR / "split_summary.csv", index=False)
feature_dictionary.to_csv(RESULTS_DIR / "feature_dictionary.csv", index=False)
initial_metrics.to_csv(RESULTS_DIR / "initial_model_metrics.csv", index=False)

validation_output = validation[["Date", "Target_End_Date", "Target_Forward_Volatility_20D", "Target_Chooser_Proxy_Price", "Current_Chooser_BSM_Price"]].copy()
for name, values in volatility_predictions.items():
    validation_output[f"A1_{name}_Volatility"] = values
for name, values in direct_predictions.items():
    validation_output[f"A2_{name}_Price"] = values
validation_output.to_csv(RESULTS_DIR / "initial_validation_predictions.csv", index=False)

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(dataset.loc[dataset["Split"] == "train", "Target_Forward_Volatility_20D"], bins=35, alpha=0.7, label="Train")
axes[0].hist(dataset.loc[dataset["Split"] == "test", "Target_Forward_Volatility_20D"], bins=35, alpha=0.7, label="Test")
axes[0].set(title="Forward Volatility Target", xlabel="Annualized volatility", ylabel="Observations")
axes[0].legend()
axes[1].plot(dataset["Date"], dataset["Target_Chooser_Proxy_Price"], color="#1f4e79", linewidth=1)
for split in ["validation", "test"]:
    axes[1].axvline(dataset.loc[dataset["Split"] == split, "Date"].iloc[0], color="black", linestyle="--", linewidth=1)
axes[1].set(title="Chooser Proxy Price and Split Boundaries", xlabel="Date", ylabel="Proxy price ($)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "targets_and_time_splits.png", dpi=180)
plt.close(fig)

print(f"Saved {len(list(RESULTS_DIR.glob('*.csv')))} Week 5 tables and {len(list(FIGURES_DIR.glob('*.png')))} figure.")
initial_metrics.head(10)

Saved 5 Week 5 tables and 1 figure.


                   Approach              Model  ...       RMSE         R2
0   1 - ML volatility + BSM      Random Forest  ...   5.372266   0.498326
1  2 - Direct proxy pricing  Linear Regression  ...   5.483655   0.477306
2   1 - ML volatility + BSM     Histogram GBDT  ...   5.499832   0.474218
3   1 - ML volatility + BSM  Linear Regression  ...   5.815108   0.412210
4   1 - ML volatility + BSM     Neural Network  ...   9.139376  -0.451911
5  2 - Direct proxy pricing     Histogram GBDT  ...   9.869078  -0.693012
6  2 - Direct proxy pricing      Random Forest  ...  10.169062  -0.797499
7  2 - Direct proxy pricing     Neural Network  ...  27.731331 -12.367444
8   1 - ML volatility + BSM      Random Forest  ...   0.074392  -0.040151
9   1 - ML volatility + BSM     Histogram GBDT  ...   0.074663  -0.047734

[10 rows x 6 columns]

# Week 5 Conclusion

The two ML architectures are connected to the existing JPM project data and chooser pricing engine. The 20-day purge prevents forward-label overlap at both chronological boundaries, and feature preprocessing remains inside each model pipeline. Week 6 will use purge-aware time-series cross-validation, select models on the validation block, evaluate the test block once, save the estimators, and explain the selected models.